In [1]:
# Cell 1 — Notebook 16: Audit saved Phi stop tokens and Qwen evidence
# CPU only. Reads existing runs and saves a separate audit.
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
from pathlib import Path
from statistics import median
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

root = Path('/content/drive/MyDrive/RPVC01/pilot')
run15 = root / 'part15_phi35_vision_qualification/runs/phi35_61143f38fe03c585'
run14 = root / 'part14_reminder_meaning_control/runs/meaning_ae09a3df2a1a542e'
close15_id = 'ee0f4463c86a19a40289305851b3365ec1907ae526a9d84f4c569f2ebb93b856'
analysis14_id = '07870ed3e52d762a486a52be26528fe8e6afd88fe03ce042a4fe80eab84816b9'
revision = '12b77fb40b63a2c73c68243d3f767aab688a1b2a'

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(',', ':'),
    ).encode('utf-8')

def digest(value):
    return hashlib.sha256(canonical(value)).hexdigest()

def sealed(path, key='data', id_key='sha256'):
    record = json.loads(path.read_text(encoding='utf-8'))
    value = record[key]
    identity = digest(value)
    if record[id_key] != identity:
        raise RuntimeError(f'Checksum mismatch: {path}')
    return value, identity

close15, found_close15 = sealed(
    run15 / 'analysis/qualification_closeout' / (close15_id[:16] + '.json'),
    'data', 'closeout_id',
)
environment, environment_id = sealed(
    run15 / 'data/environment_freeze.json', 'environment', 'environment_id',
)
preflight, preflight_id = sealed(
    run15 / 'data/model_preflight.json', 'binding', 'preflight_id',
)
manifest, manifest_id = sealed(run15 / 'execution/generation_manifest.json')
prep_id = close15['preparation_id']
prepared, found_prep = sealed(
    run15 / 'analysis/preparation' / prep_id[:16] / 'prepared_responses.json',
    'data', 'preparation_id',
)
if (
    found_close15 != close15_id or close15['qualification'] != 'FAIL'
    or close15['status'] != 'PHI_BENIGN_QUALIFICATION_CLOSED'
    or found_prep != prep_id or environment['model_revision'] != revision
    or preflight['model_revision'] != revision
    or environment['input_id'] != close15['input_id']
    or any(
        prepared[key] != close15[key]
        for key in ('input_id', 'preflight_id', 'manifest_id')
    )
    or manifest['environment_id'] != preflight['environment_id']
    or preflight['environment_id'] != environment_id
    or manifest['preflight_id'] != preflight_id
    or preflight_id != close15['preflight_id']
    or manifest_id != close15['manifest_id']
    or manifest['generation_config'] != preflight['generation_config']
):
    raise RuntimeError('Phi audit source records do not agree.')

# Marker IDs were read from this exact pinned tokenizer configuration.
tokenizer_config_sha = '581d4654c018fbcef2504b57a805dcad37126038137b5d25d22b5a02abc1ef29'
if environment['evidence']['metadata_sha256']['tokenizer_config.json'] != tokenizer_config_sha:
    raise RuntimeError('Tokenizer configuration differs from the audited upstream file.')

marker_ids = {'tokenizer_eos': 32000, 'chat_turn_end': 32007}
config = manifest['generation_config']
eos = config['eos_token_id']
eos_ids = [eos] if isinstance(eos, int) else sorted(eos or [])
cap = config['max_new_tokens']
rows = prepared['rows']
if (
    cap != 256 or len(rows) != 60 or len({r['job_id'] for r in rows}) != 60
    or set(manifest['job_ids']) != {r['job_id'] for r in rows}
):
    raise RuntimeError('Unexpected Phi generation design.')

def inspect_tokens(row):
    jid = row['job_id']
    raw, raw_id = sealed(run15 / 'execution/raw_tokens' / (jid + '.json'))
    tokens = raw['generated_token_ids']
    receipt = raw['receipt']
    if (
        raw_id != row['raw_sha256'] or raw['job_id'] != jid
        or digest(receipt) != row['receipt_sha256']
        or receipt['attempt_id'] != row['attempt_id']
        or receipt['manifest_id'] != manifest_id
        or not 1 <= len(tokens) <= cap
        or any(type(token) is not int or token < 0 for token in tokens)
    ):
        raise RuntimeError(f'Invalid original tokens: {jid}')

    counts = Counter(tokens)
    first = {
        name: tokens.index(token) + 1 if counts[token] else None
        for name, token in marker_ids.items()
    }
    return {
        'job_id': jid, 'order': row['order'], 'raw_sha256': raw_id,
        'generated_tokens': len(tokens),
        'configured_stop_hits': sum(counts[t] for t in eos_ids),
        'hit_cap_without_terminal_eos': len(tokens) == cap and tokens[-1] not in eos_ids,
        'first_marker_position_1_based': first,
    }

print('Checking original Phi token sequences...', flush=True)
with ThreadPoolExecutor(max_workers=6) as pool:
    token_rows = list(pool.map(inspect_tokens, rows))

capped = sum(r['hit_cap_without_terminal_eos'] for r in token_rows)
if capped != close15['possibly_truncated_responses']:
    raise RuntimeError('Saved token lengths disagree with the qualification closeout.')

marker_summary = {}
for name, token in marker_ids.items():
    positions = [r['first_marker_position_1_based'][name] for r in token_rows]
    observed = [p for p in positions if p is not None]
    marker_summary[name] = {
        'token_id': token,
        'configured_as_stop': token in eos_ids,
        'responses_with_marker': len(observed),
        'responses_with_marker_and_later_tokens': sum(
            p is not None and p < r['generated_tokens']
            for r, p in zip(token_rows, positions)
        ),
        'median_first_position_1_based': median(observed) if observed else None,
    }

analysis14 = json.loads((
    run14 / 'analysis/assistant_analysis' / analysis14_id[:16] / 'results.json'
).read_text())
if digest(analysis14) != analysis14_id:
    raise RuntimeError('Notebook 14 analysis hash mismatch.')

closeouts14 = []
for path in sorted((run14 / 'analysis/closeout').glob('*.json')):
    item = json.loads(path.read_text())
    if item.get('analysis_id') == analysis14_id:
        identity = digest(item)
        if path.stem != identity[:16] or item['status'] != 'NOTEBOOK14_ANALYSIS_CLOSED':
            raise RuntimeError(f'Notebook 14 closeout did not verify: {path.name}')
        closeouts14.append((identity, item))
if len(closeouts14) != 1:
    raise RuntimeError('Expected one verified Notebook 14 closeout.')

close14_id, close14 = closeouts14[0]
primary14 = analysis14['results']['primary_rubric']
report = {
    'phi': {
        'closeout_id': close15_id, 'preflight_id': preflight_id,
        'manifest_id': manifest_id, 'environment_id': environment_id,
        'tokenizer_config_sha256': tokenizer_config_sha, 'model_revision': revision,
        'configured_eos_ids': eos_ids, 'max_new_tokens': cap,
        'cap_without_terminal_eos': capped,
        'configured_stop_hits': sum(r['configured_stop_hits'] for r in token_rows),
        'markers': marker_summary, 'rows': token_rows,
        'gates': close15['gates'], 'qualification': close15['qualification'],
    },
    'qwen_notebook14': {
        'analysis_id': analysis14_id, 'closeout_id': close14_id,
        'effects': primary14['effects'],
        'utility_gate': primary14['utility_gate'],
        'transcript_gate': primary14['transcript_gate'],
        'failed_transcript_conditions': close14['failed_transcript_cells'],
        'transcript_counts': close14['transcript_counts'],
        'mismatch_causes': close14['mismatch_causes'],
        'qualification': primary14['qualification'],
    },
    'scope': 'Post-hoc inspection of original token IDs and frozen records.',
    'new_generation_attempts': 0,
    'status': 'CPU_EVIDENCE_AUDIT_SAVED',
}

audit_id = digest(report)
path = root / 'part16_evidence_audit' / (audit_id[:16] + '.json')
path.parent.mkdir(parents=True, exist_ok=True)
content = canonical({'audit_id': audit_id, 'data': report})
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError('Existing audit differs.')
else:
    temporary = path.with_suffix('.tmp')
    temporary.write_bytes(content)
    os.replace(temporary, path)

print('Phi configured stop IDs:', eos_ids, '| cap without terminal EOS:', f'{capped}/60')
print('Phi configured stop hits:', report['phi']['configured_stop_hits'])
for name, summary in marker_summary.items():
    print(name, summary)
print('Notebook 14 utility / transcript:', primary14['utility_gate'], '/', primary14['transcript_gate'])
print('Notebook 14 failed transcript conditions:', close14['failed_transcript_cells'])
print('Notebook 14 primary effect:', primary14['effects']['primary_controlled_order_interaction'])
print('Audit ID:', audit_id)
print('Status:', report['status'])
print('New generation attempts: 0')

Mounted at /content/drive
Checking original Phi token sequences...
Phi configured stop IDs: [2] | cap without terminal EOS: 60/60
Phi configured stop hits: 0
tokenizer_eos {'token_id': 32000, 'configured_as_stop': False, 'responses_with_marker': 60, 'responses_with_marker_and_later_tokens': 60, 'median_first_position_1_based': 4.0}
chat_turn_end {'token_id': 32007, 'configured_as_stop': False, 'responses_with_marker': 60, 'responses_with_marker_and_later_tokens': 60, 'median_first_position_1_based': 3.0}
Notebook 14 utility / transcript: PASS / FAIL
Notebook 14 failed transcript conditions: {'neutral_mention | neutral_text_control | answer_first': {'correct_or_exact': 35, 'count_bounds': [35, 35], 'gate': 'FAIL', 'n': 40, 'required': 36, 'unclear': 0, 'wrong_or_not_exact': 5}, 'neutral_mention | neutral_text_control | transcribe_first': {'correct_or_exact': 12, 'count_bounds': [12, 12], 'gate': 'FAIL', 'n': 40, 'required': 36, 'unclear': 0, 'wrong_or_not_exact': 28}}
Notebook 14 primar